# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Get metadata object (not as dict)
metadata = dataset.metadata
print(f"Dataset name: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

`mlcroissant` organizes source data into record sets, each of which may link to one or more files. Each record set contains fields, each mapping to a column. All entities are referenced by their `@id`.

In [ ]:
# List all available record sets by their @id
record_sets = dataset.record_sets
print("Available record sets and their @id:")
for rs in record_sets:
    print(f"  - @id: {getattr(rs, '@id', None)} | name: {getattr(rs, 'name', None)}")

# Show all fields (columns) for each record set with their @id
for rs in record_sets:
    print(f"\nRecord Set: {getattr(rs, 'name', None)} (@id: {getattr(rs, '@id', None)})")
    for field in getattr(rs, 'fields', []):
        print(f"  Field '@id': {getattr(field, '@id', None)} | name: {getattr(field, 'name', None)} | dataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demo: use the first available record set
record_sets = dataset.record_sets

record_set_ids = [getattr(rs, '@id') for rs in record_sets]
# If more than one record set, print all
print(f"Record set '@id's: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading data for record set '@id': {record_set_id}")
    recs = dataset.records(record_set=record_set_id)
    recs_list = list(recs)
    df = pd.DataFrame(recs_list)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis. References always use the `@id` of the field and record set.

In [ ]:
# === Example: select a numeric field to filter/normalize ===
# Use the first record set and its numeric fields
first_record_set = dataset.record_sets[0]
first_record_set_id = getattr(first_record_set, '@id')
df = dataframes[first_record_set_id]

# Get the numeric fields in this record set (by data type)
numeric_fields = [getattr(f, '@id') for f in first_record_set.fields if getattr(f, 'data_type', None) in ['Number', 'Float', 'Integer']]
print("Numeric fields available (by @id):", numeric_fields)
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    raise Exception("No numeric fields detected.")

threshold = df[numeric_field_id].quantile(0.5) if df[numeric_field_id].dtype.kind in 'fi' else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a group field if present (categorical)
all_field_ids = [getattr(f, '@id') for f in first_record_set.fields]
potential_group_fields = [fid for fid in all_field_ids if df[fid].nunique() < max(10, df.shape[0]//10)]
group_field = potential_group_fields[0] if potential_group_fields else None

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped data mean '{numeric_field_id}' by '{group_field}':")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple visualization for the selected numeric field
import matplotlib.pyplot as plt

if len(filtered_df) > 0:
    plt.figure(figsize=(8,4))
    plt.hist(filtered_df[numeric_field_id], bins=30, alpha=0.8)
    plt.title(f"Distribution of '{numeric_field_id}' after filtering")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    plt.figure(figsize=(8,4))
    plt.hist(filtered_df[f"{numeric_field_id}_normalized"], bins=30, alpha=0.7, color='orange')
    plt.title(f"Distribution of Normalized '{numeric_field_id}'")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.ylabel('Count')
    plt.show()
else:
    print("No records to visualize after filtering.")

## 6. Conclusion
In this notebook, we have demonstrated how to load, explore, and process the Mass Spectrometry dataset on extracellular vesicles using `mlcroissant`.

Key steps performed:
- Loaded metadata and listed record sets (`@id`s).
- Loaded data to pandas `DataFrame` for each record set by its unique `@id`.
- Performed filtering, normalization, and optional grouping using field `@id` references.
- Visualized distributions for a selected numeric field.

Continue your domain-specific EDA and modeling using the referenced field and record set `@id` values for reproducibility.